In [1]:
!pip install rdflib

In [2]:
import rdflib
from rdflib import Graph, Literal, URIRef
from rdflib.namespace import RDF, XSD
from datetime import datetime, timedelta
import random

# Define namespaces
ex = rdflib.Namespace("http://example.org/")
health = rdflib.Namespace("http://example.org/health/")
finance = rdflib.Namespace("http://example.org/finance/")
general = rdflib.Namespace("http://example.org/general/")
time = rdflib.Namespace("http://example.org/time/")

# Create an empty graph
g = Graph()

# Bind namespaces to prefixes for better readability in the output
g.bind("ex", ex)
g.bind("health", health)
g.bind("finance", finance)
g.bind("general", general)
g.bind("time", time)

# Set the start date for our simulation
start_date = datetime(2024, 10, 20, 10, 0, 0) # October 20, 2024, 10:00 AM IST
num_months = 6
current_date = start_date

# Lists of entities and relations for each domain
persons = [ex["JohnDoe"], ex["JaneSmith"], ex["Robert"], ex["Alice"]]
diseases = [health["Flu"], health["COVID19"], health["Allergy"]]
symptoms = [health["Fever"], health["Cough"], health["SoreThroat"], health["Fatigue"], health["RunnyNose"]]
medications = [health["Paracetamol"], health["Ibuprofen"], health["Antihistamine"]]
doctors = [health["DrAlice"], health["DrBob"]]
hospitals = [health["CityHospital"], health["GeneralClinic"]]

accounts = [finance["JohnChecking"], finance["JaneSavings"], finance["RobertCredit"]]
transactions = [finance["T1"], finance["T2"], finance["T3"], finance["T4"], finance["T5"]]
financial_products = [finance["StockA"], finance["BondB"], finance["MutualFundC"]]
banks = [finance["CityBank"], finance["NationalCreditUnion"]]

locations = [general["Home"], general["Office"], general["Gym"], general["Restaurant"]]
activities = [general["Meeting"], general["Workout"], general["Lunch"], general["Travel"]]
items = [general["BookA"], general["Laptop"], general["Phone"]]

relations_health = [health["hasSymptom"], health["diagnosedWith"], health["prescribed"], health["visitedDoctor"], health["atHospital"]]
relations_finance = [finance["hasAccount"], finance["madeTransaction"], finance["investedIn"], finance["accountAt"], finance["transactionAmount"]]
relations_general = [general["locatedAt"], general["participatedIn"], general["ownsItem"]]

# Function to add a temporal triple
def add_temporal_triple(subject, predicate, object, timestamp):
    g.add((subject, predicate, object))
    g.add((subject, time["timestamp"], Literal(timestamp.isoformat(), datatype=XSD.dateTime)))

# Simulate events over 6 months
for month in range(num_months):
    days_in_month = (current_date.replace(month=current_date.month % 12 + 1, day=1) - timedelta(days=1)).day
    for day in range(days_in_month):
        for hour in range(random.randint(1, 5)): # Simulate a few events per day
            minute = random.randint(0, 59)
            second = random.randint(0, 59)
            event_time = current_date.replace(day=day + 1, hour=random.randint(8, 18), minute=minute, second=second)

            # Simulate Health events
            if random.random() < 0.15:
                person = random.choice(persons)
                if random.random() < 0.4:
                    disease_or_symptom = random.choice(diseases + symptoms)
                    relation = random.choice([health["hasSymptom"], health["diagnosedWith"]])
                    add_temporal_triple(person, relation, disease_or_symptom, event_time)
                elif random.random() < 0.3:
                    person = random.choice(persons)
                    med = random.choice(medications)
                    doctor = random.choice(doctors)
                    add_temporal_triple(doctor, health["prescribed"], med, event_time)
                    add_temporal_triple(person, health["prescribed"], med, event_time)
                    add_temporal_triple(person, health["visitedDoctor"], doctor, event_time)
                elif random.random() < 0.2:
                    person = random.choice(persons)
                    hospital = random.choice(hospitals)
                    add_temporal_triple(person, health["atHospital"], hospital, event_time)

            # Simulate Finance events
            if random.random() < 0.2:
                person = random.choice(persons)
                account = random.choice(accounts)
                relation_fin = random.choice(relations_finance[:-1]) # Exclude transactionAmount initially
                add_temporal_triple(person, finance["hasAccount"], account, event_time)
                if random.random() < 0.3:
                    transaction = random.choice(transactions)
                    add_temporal_triple(account, finance["madeTransaction"], transaction, event_time)
                    amount = round(random.uniform(10, 1000), 2)
                    add_temporal_triple(transaction, finance["transactionAmount"], Literal(amount, datatype=XSD.float), event_time)
                elif random.random() < 0.1:
                    product = random.choice(financial_products)
                    add_temporal_triple(person, finance["investedIn"], product, event_time)
                elif random.random() < 0.2:
                    bank = random.choice(banks)
                    add_temporal_triple(account, finance["accountAt"], bank, event_time)

            # Simulate General events
            if random.random() < 0.3:
                person = random.choice(persons)
                location = random.choice(locations)
                add_temporal_triple(person, general["locatedAt"], location, event_time)
            elif random.random() < 0.25:
                person = random.choice(persons)
                activity = random.choice(activities)
                add_temporal_triple(person, general["participatedIn"], activity, event_time)
            elif random.random() < 0.1:
                person = random.choice(persons)
                item = random.choice(items)
                add_temporal_triple(person, general["ownsItem"], item, event_time)

    current_date += timedelta(days=30) # Approximate month increment


In [4]:

# Serialize the graph to a TTL file
g.serialize(destination="simulated_tkg.ttl", format="turtle")

print("Simulated TKG created and saved as simulated_tkg.ttl")

In [3]:
!pip install pyvis

In [4]:
from pyvis.network import Network
from IPython.core.display import display, HTML
from IPython.display import IFrame

net = Network(height='600px', width='100%', notebook=True)
for s, p, o in g:
    net.add_node(str(s), label=str(s))
    net.add_node(str(o), label=str(o))
    net.add_edge(str(s), str(o), label=str(p))

# net.show("rdf_graph.html")
# Save and load in iframe
net.save_graph("rdf_graph.html")

In [5]:
from rdflib import Graph, URIRef
from datetime import datetime
from collections import defaultdict
from pyvis.network import Network
import random

# Define namespaces (assuming these are bound in your loaded graph 'g')
ex = rdflib.Namespace("http://example.org/")
health = rdflib.Namespace("http://example.org/health/")
finance = rdflib.Namespace("http://example.org/finance/")
general = rdflib.Namespace("http://example.org/general/")
time = rdflib.Namespace("http://example.org/time/")

def visualize_whole_tkg_loaded(graph, output_filename="whole_tkg.html"):
    """Visualizes the entire TKG from a loaded rdflib Graph."""
    net = Network(notebook=True, height="750px", width="100%", directed=True)
    for s, p, o in graph:
        s_label = s.split("/")[-1] if "/" in str(s) else str(s)
        o_label = o.split("/")[-1] if "/" in str(o) else str(o)
        net.add_node(str(s), label=s_label)
        net.add_node(str(o), label=o_label)
        p_label = p.split("/")[-1] if "/" in str(p) else str(p)
        net.add_edge(str(s), str(o), label=p_label)
    # net.show(output_filename)
    print(f"Whole TKG visualization saved to {output_filename}")
    return net

def visualize_tkg_per_month_loaded(graph, output_prefix="tkg_month_", color_map=None):
    """Visualizes the TKG for each month with different colors from a loaded graph.
       Returns a dictionary of Network objects, keyed by month."""
    temporal_data = []
    timestamps = set()
    time_property = URIRef("http://example.org/time/timestamp")

    for s, p, o in graph:
        timestamp_literals = list(graph.objects(s, time_property)) + list(graph.objects(o, time_property))
        for ts_lit in timestamp_literals:
            try:
                timestamp = datetime.fromisoformat(str(ts_lit).replace('Z', '+00:00'))
                temporal_data.append((s, p, o, timestamp))
                timestamps.add(timestamp)
            except ValueError:
                pass

    if color_map is None:
        colors = ["#a8dadc", "#457b9d", "#1d3557", "#e63946", "#f1faee", "#a8dadc"]
        color_map = {i: colors[i % len(colors)] for i in range(6)} # Assuming max 6 months

    monthly_networks = {}
    monthly_data = defaultdict(list)
    for s, p, o, ts in temporal_data:
        month_year = ts.strftime("%Y-%m")
        monthly_data[month_year].append((s, p, o))

    sorted_months = sorted(monthly_data.keys())

    for i, month in enumerate(sorted_months):
        net = Network(notebook=True, height="750px", width="100%", directed=True)
        nodes = set()
        for s, p, o in monthly_data[month]:
            s_str = str(s)
            o_str = str(o)
            if s_str not in nodes:
                net.add_node(s_str, label=s.split("/")[-1] if "/" in str(s) else str(s), color=color_map.get(i, "gray"))
                nodes.add(s_str)
            if o_str not in nodes:
                net.add_node(o_str, label=o.split("/")[-1] if "/" in str(o) else str(o), color=color_map.get(i, "gray"))
                nodes.add(o_str)
            p_label = p.split("/")[-1] if "/" in str(p) else str(p)
            net.add_edge(s_str, o_str, label=p_label, color=color_map.get(i, "lightgray"))

        output_filename = f"tkg_month_{month}.html"
        # net.show(output_filename) # Keep showing for immediate viewing
        print(f"Visualization for {month} saved to {output_filename}")
        monthly_networks[month] = net # Return the Network object

    return monthly_networks

if __name__ == "__main__":
    # Assume your graph is already loaded into a variable named 'g'
    # For demonstration purposes, let's simulate loading from a file here
    # g = Graph()
    # try:
    #     g.parse("simulated_tkg.ttl", format="turtle")
    # except FileNotFoundError:
    #     print("Error: simulated_tkg.ttl not found. Please run the TKG creation script first.")
    #     exit()

    # Visualize the whole TKG from the loaded graph 'g'
    visualize_whole_tkg_loaded(g)
    net.save_graph(f"rdf_graph_whole6mon.html")

    # Visualize per (simulated) month from the loaded graph 'g' and save the Network objects
    monthly_nets = visualize_tkg_per_month_loaded(g)

    # Now you can iterate through the monthly_nets dictionary and save each graph
    for month, net in monthly_nets.items():
        net.save_graph(f"rdf_graph_monthwise_{month}.html")
        print(f"Month-wise graph for {month} saved to rdf_graph_monthwise_{month}.html")

Whole TKG visualization saved to whole_tkg.html
Visualization for 2024-10 saved to tkg_month_2024-10.html
Visualization for 2024-11 saved to tkg_month_2024-11.html
Visualization for 2024-12 saved to tkg_month_2024-12.html
Visualization for 2025-01 saved to tkg_month_2025-01.html
Visualization for 2025-02 saved to tkg_month_2025-02.html
Visualization for 2025-03 saved to tkg_month_2025-03.html
Month-wise graph for 2024-10 saved to rdf_graph_monthwise_2024-10.html
Month-wise graph for 2024-11 saved to rdf_graph_monthwise_2024-11.html
Month-wise graph for 2024-12 saved to rdf_graph_monthwise_2024-12.html
Month-wise graph for 2025-01 saved to rdf_graph_monthwise_2025-01.html
Month-wise graph for 2025-02 saved to rdf_graph_monthwise_2025-02.html
Month-wise graph for 2025-03 saved to rdf_graph_monthwise_2025-03.html
